# Решения: Интеграл как накопление: площадь под кривой loss

**Для преподавателя.** Ниже по разделам разобраны все задачи `lesson.ipynb` и `homework.ipynb`. Не выдавать до сдачи.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def curve(x):
    return 1.2 + 0.4 * x + 0.2 * np.sin(1.3 * x)

left, right = 0.0, 6.0


## Урок. 1. Левые прямоугольники

In [ ]:
def left_rectangle_area(fun, left, right, parts):
    dx = (right - left) / parts
    points = left + np.arange(parts) * dx
    return float(np.sum(fun(points)) * dx)

area_10 = left_rectangle_area(curve, left, right, 10)
assert 0 < area_10 < 30
print(round(area_10, 4))


## Урок. 2. Трапеции без готовой функции

In [ ]:
def trapezoid_area(fun, left, right, parts):
    points = np.linspace(left, right, parts + 1)
    values = fun(points)
    dx = (right - left) / parts
    return float(dx * (0.5 * values[0] + np.sum(values[1:-1]) + 0.5 * values[-1]))

trap_10 = trapezoid_area(curve, left, right, 10)
assert 0 < trap_10 < 30
print(round(trap_10, 4))


## Урок. 3. Сходимость оценки площади

In [ ]:
parts_values = [5, 10, 20, 100]
left_estimates = [left_rectangle_area(curve, left, right, n) for n in parts_values]
trap_estimates = [trapezoid_area(curve, left, right, n) for n in parts_values]
assert len(left_estimates) == len(trap_estimates) == len(parts_values)
print([round(x, 5) for x in left_estimates])
print([round(x, 5) for x in trap_estimates])


## Урок. 4. Визуализация накопления

In [ ]:
grid = np.linspace(left, right, 101)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid, curve(grid), color="navy")
ax.fill_between(grid, 0, curve(grid), alpha=0.25)
ax.set(xlabel="x", ylabel="curve(x)", title="Площадь как накопленная величина")
assert len(ax.lines) >= 1 and len(ax.collections) >= 1
plt.show()


## Урок. 5. Площадь под историей loss

In [ ]:
epoch_loss = np.array([2.4, 2.1, 1.9, 1.7, 1.55, 1.45, 1.38, 1.34, 1.31, 1.29])
epochs = np.arange(len(epoch_loss), dtype=float)
accumulated_loss = float(np.trapezoid(epoch_loss, epochs))
assert accumulated_loss > 0
print(round(accumulated_loss, 4))


## Урок. 6. Финальный loss и весь путь — разные критерии

In [ ]:
loss_a = np.array([2.4, 2.1, 1.8, 1.55, 1.35, 1.2, 1.1, 1.04, 1.01, 1.0])
loss_b = np.array([2.4, 1.7, 1.3, 1.1, 1.02, 0.99, 0.98, 0.98, 0.99, 1.02])
final_winner = "A" if loss_a[-1] < loss_b[-1] else "B"
area_a = float(np.trapezoid(loss_a, epochs))
area_b = float(np.trapezoid(loss_b, epochs))
area_winner = "A" if area_a < area_b else "B"
assert final_winner != area_winner
print(final_winner, area_winner, round(area_a, 3), round(area_b, 3))


## Урок. 7. Нормировка по времени

In [ ]:
short_loss = loss_b[:5]
raw_short = float(np.trapezoid(short_loss))
mean_short = raw_short / (len(short_loss) - 1)
mean_full = float(np.trapezoid(loss_b)) / (len(loss_b) - 1)
assert raw_short > 0 and mean_short > 0 and mean_full > 0
print(round(raw_short, 4), round(mean_short, 4), round(mean_full, 4))


## Урок. 8. Самостоятельно: границы метрики

In [ ]:
INTEGRAL_NOTE = (
    "Площадь под кривой loss описывает весь путь обучения и полезна для сравнения скорости "
    "снижения ошибки при одинаковых эпохах и стоимости шага. Она не заменяет финальный loss: "
    "модель с меньшей площадью может закончить хуже, а длина запуска напрямую меняет сырую площадь. "
    "Поэтому сравнивать нужно одинаковые интервалы или нормированную площадь. Это диагностическая "
    "метрика эксперимента, а не функция потерь, которую модель обязательно минимизирует."
)
assert len(INTEGRAL_NOTE) >= 260
print(INTEGRAL_NOTE)


## ДЗ. Данные и функции

In [ ]:
import numpy as np

def trapezoid_from_values(values, step=1.0):
    values = np.asarray(values, dtype=float)
    return float(step * (0.5 * values[0] + np.sum(values[1:-1]) + 0.5 * values[-1]))


## ДЗ. Закрепление: площадь по таблице

In [ ]:
x = np.linspace(0.0, 5.0, 101)
y = 1.0 + 0.3 * x + 0.15 * np.cos(1.7 * x)
area = trapezoid_from_values(y, step=x[1] - x[0])
assert 0 < area < 20
print(round(area, 4))


## ДЗ. База: сравнение историй

In [ ]:
loss_c = np.array([3.0, 2.7, 2.5, 2.35, 2.2, 2.1, 2.0, 1.94, 1.9, 1.86])
loss_d = np.array([3.0, 2.6, 2.35, 2.2, 2.08, 1.98, 1.9, 1.83, 1.79, 1.76])
areas = [trapezoid_from_values(loss_c), trapezoid_from_values(loss_d)]
winner = "C" if areas[0] < areas[1] else "D"
assert winner == "D"
print(areas, winner)


## ДЗ. Углубление: честное сравнение разной длины

In [ ]:
short_run = loss_d[:6]
long_run = loss_d
normalized = [
    trapezoid_from_values(short_run) / (len(short_run) - 1),
    trapezoid_from_values(long_run) / (len(long_run) - 1),
]
assert len(normalized) == 2
print(normalized)


## ДЗ. Вызов: контрпример для площади

In [ ]:
COUNTEREXAMPLE = (
    "Например, история P = [5, 1, 1, 1] имеет площадь 5, а финальный loss 1. "
    "История Q = [2, 2, 2, 0.5] имеет площадь 5.25, но заканчивает с меньшим loss 0.5. "
    "По площади лучше P, по финальному качеству лучше Q. Контрпример показывает, что критерии "
    "отвечают на разные вопросы: площадь — о затратах ошибки на пути, финальное значение — "
    "о результате последнего шага. Выбор критерия должен следовать задаче."
)
READY = True
assert len(COUNTEREXAMPLE) >= 260 and READY
print(COUNTEREXAMPLE)
